# 07 — Validate routing outputs

Validate source coverage, graph status, supported schemas, non-negative and nested car/pedestrian accessibility contours, PT zero-stop flags, lagged-stock provenance, and cumulative firm accessibility against the long Fachgruppe table.

In [ ]:
from pathlib import Path
import hashlib
import json
import sys

import numpy as np
import pandas as pd

def discover_project_dir() -> Path:
    for candidate in [Path.cwd(), *Path.cwd().parents]:
        if (candidate / "ANAL").is_dir() and (candidate / "OGD").is_dir():
            return candidate
    raise RuntimeError("Run this notebook from the project directory or a subdirectory containing ANAL/ and OGD/.")

PROJECT_DIR = discover_project_dir()
ANAL_DATA = PROJECT_DIR / "ANAL" / "data"
ROUTING_DATA = ANAL_DATA / "routing"
POI_DIR = ROUTING_DATA / "destinations"
PBF_DIR = PROJECT_DIR / "OGD" / "GIP"
PANEL_PATH = ANAL_DATA / "raster_quarter_panel_100m.parquet"
ACTIVE_CELLS_PATH = ROUTING_DATA / "inputs" / "active_routing_cells_100m.parquet"
STATUS_PATH = ROUTING_DATA / "status" / "routing_feature_status.csv"
GRAPH_STATUS_PATH = ROUTING_DATA / "status" / "graph_build_status.csv"
SKIPPED_ACCESSIBILITY_PATH = ROUTING_DATA / "status" / "accessibility_skipped_origins.csv"
REPORT_PATH = ROUTING_DATA / "reports" / "routing_validation_summary.csv"
WORK_ROOT = ROUTING_DATA / "work" / "routing_features"
GRAPH_VALHALLA_IMAGE = "ghcr.io/valhalla/valhalla-scripted:3.8.3"
GRAPH_VALHALLA_IMAGE_ID = "sha256:24ef7955899dececb94e26c6dfb89d64fabfae875f980432694b0261eb6c251b"
YEARS = range(2015, 2026)
sys.path.insert(0, str(PROJECT_DIR / "ANAL" / "routing"))
from routing_utils import ACCESS_MINUTES, fachgruppe_ids, main_access_columns
FACHGRUPPE_IDS = fachgruppe_ids(PANEL_PATH)
REPORT_PATH.parent.mkdir(parents=True, exist_ok=True)

In [ ]:
def result(year, product, check, passed):
    return {"year": year, "product": product, "check": check, "status": "OK" if bool(passed) else "CHECK"}

skipped_accessibility = pd.read_csv(SKIPPED_ACCESSIBILITY_PATH) if SKIPPED_ACCESSIBILITY_PATH.exists() else pd.DataFrame(columns=["year", "grid_id"])

def exclude_documented_skips(frame, year):
    skipped_ids = set(skipped_accessibility.loc[skipped_accessibility["year"] == year, "grid_id"].astype(str))
    grid_column = "grid_id_100m" if "grid_id_100m" in frame else "grid_id"
    return frame.loc[~frame[grid_column].astype(str).isin(skipped_ids)] if grid_column in frame else frame

def numeric_checks(frame, columns, year, product):
    frame = exclude_documented_skips(frame, year)
    rows = []
    for column in columns:
        values = pd.to_numeric(frame[column], errors="coerce") if column in frame else pd.Series(dtype=float)
        rows += [result(year, product, f"{column} numeric", len(values) == len(frame) and values.notna().all()), result(year, product, f"{column} non-negative", (values >= 0).all())]
    contour_sets = {
        "pop_access": ACCESS_MINUTES, "existing_firms_access": ACCESS_MINUTES, "same_fachgruppe_firms_access": ACCESS_MINUTES,
        "walk_pop": (5, 10), "walk_firms": (5, 10), "walk_same_fachgruppe_firms": (5, 10),
        "walk_pt_stops": (5, 10), "walk_pt_departures": (5, 10), "walk_pt_routes": (5, 10),
        "reachable_cells": ACCESS_MINUTES, "reachable_cells_walk": (5, 10),
    }
    for prefix, minutes_set in contour_sets.items():
        for left_minutes, right_minutes in zip(minutes_set, minutes_set[1:]):
            left, right = f"{prefix}_{left_minutes}min", f"{prefix}_{right_minutes}min"
            if {left, right}.issubset(frame): rows.append(result(year, product, f"{left} <= {right}", (frame[left] <= frame[right]).all()))
    return rows

records = []
for source_path in [PANEL_PATH, ACTIVE_CELLS_PATH, ANAL_DATA / "firms_assigned_100m.geoparquet"]:
    records.append(result("all", "source", f"{source_path.name} exists", source_path.exists()))
active_count = len(pd.read_parquet(ACTIVE_CELLS_PATH, columns=["grid_id"]))
for year in YEARS:
    records.append(result(year, "source", "yearly destinations exist", (POI_DIR / f"austria-{year}-pois.geoparquet").exists()))
    feature_dir = ROUTING_DATA / "features" / str(year)
    nearest_path = feature_dir / "nearest_infrastructure_100m.parquet"
    main_path = feature_dir / "accessibility_potentials_100m.parquet"
    pedestrian_path = feature_dir / "pedestrian_accessibility_quarter_100m.parquet"
    long_path = feature_dir / "fachgruppe_accessibility_quarter_100m.parquet"
    firm_path = feature_dir / "firm_accessibility_quarter_100m.parquet"
    for product, path in {"nearest": nearest_path, "grid-quarter": main_path, "pedestrian-grid-quarter": pedestrian_path, "Fachgruppe-long": long_path, "firm-quarter": firm_path}.items():
        records.append(result(year, product, "canonical output exists", path.exists()))
    stale = list(feature_dir.glob("*.partial*")) + list(feature_dir.glob("*.slice*")) + list(feature_dir.glob("*.tmp")) + list(feature_dir.glob(".accessibility_parts*"))
    records.append(result(year, "cleanup", "no visible routing intermediates", not stale))
    records.append(result(year, "cleanup", "no hidden checkpoint directory", not (WORK_ROOT / str(year)).exists()))
    if nearest_path.exists():
        nearest = pd.read_parquet(nearest_path)
        records += [result(year, "nearest", "one row per active cell", len(nearest) == active_count), result(year, "nearest", "grid_id unique", nearest.grid_id.is_unique)]
    if main_path.exists():
        main = pd.read_parquet(main_path)
        main_numeric = [*main_access_columns(), *[f"reachable_cells_{minutes}min" for minutes in ACCESS_MINUTES]]
        required = {"grid_id", "year", "quarter", *main_numeric}
        records += [result(year, "grid-quarter", "required schema", required.issubset(main)), result(year, "grid-quarter", "year and four quarters complete", set(main.year) == {year} and set(main.quarter) == {1,2,3,4} and len(main) == active_count * 4), result(year, "grid-quarter", "unique grid-year-quarter keys", not main.duplicated(["grid_id", "year", "quarter"]).any()), result(year, "grid-quarter", "no contemporaneous or Sparte fields", not any(c == "active_firms_t" or "sparte_" in c.lower() for c in main.columns))]
        records += numeric_checks(main, main_numeric, year, "grid-quarter")
    if pedestrian_path.exists():
        pedestrian = pd.read_parquet(pedestrian_path)
        pedestrian_numeric = [column for minutes in (5, 10) for column in (f"walk_pop_{minutes}min", f"walk_firms_{minutes}min", f"walk_pt_stops_{minutes}min", f"walk_pt_departures_{minutes}min", f"walk_pt_routes_{minutes}min", f"reachable_cells_walk_{minutes}min")]
        required = {"grid_id", "year", "quarter", "pt_ohne_haltestelle", *pedestrian_numeric}
        records += [result(year, "pedestrian-grid-quarter", "required schema", required.issubset(pedestrian)), result(year, "pedestrian-grid-quarter", "year and four quarters complete", set(pedestrian.year) == {year} and set(pedestrian.quarter) == {1,2,3,4} and len(pedestrian) == active_count * 4), result(year, "pedestrian-grid-quarter", "unique grid-year-quarter keys", not pedestrian.duplicated(["grid_id", "year", "quarter"]).any()), result(year, "pedestrian-grid-quarter", "zero-stop flag matches 10-minute stop count", (pedestrian["pt_ohne_haltestelle"] == pedestrian["walk_pt_stops_10min"].eq(0).astype(int)).all())]
        records += numeric_checks(pedestrian, pedestrian_numeric, year, "pedestrian-grid-quarter")
    if long_path.exists():
        long = pd.read_parquet(long_path)
        long_numeric = [*[f"same_fachgruppe_firms_access_{minutes}min" for minutes in ACCESS_MINUTES], *[f"walk_same_fachgruppe_firms_{minutes}min" for minutes in (5, 10)]]
        required = {"grid_id", "year", "quarter", "Fachgruppe_ID", "own_cell_same_fachgruppe_firms", "own_cell_walk_same_fachgruppe_firms", *long_numeric}
        coverage = long.groupby(["grid_id", "year", "quarter"])["Fachgruppe_ID"].nunique()
        records += [result(year, "Fachgruppe-long", "required schema", required.issubset(long)), result(year, "Fachgruppe-long", "all 95 Fachgruppen per grid-quarter", set(long.Fachgruppe_ID.astype(str)) == set(FACHGRUPPE_IDS) and (coverage == 95).all()), result(year, "Fachgruppe-long", "unique keys", not long.duplicated(["grid_id","year","quarter","Fachgruppe_ID"]).any())]
        records += numeric_checks(long, long_numeric, year, "Fachgruppe-long")
    if firm_path.exists() and long_path.exists() and main_path.exists():
        firm = pd.read_parquet(firm_path)
        required = {"firm_id", "grid_id_100m", "Fachgruppe_ID", "year", "quarter", "included_in_lagged_stock", *[f"same_fachgruppe_firms_access_{minutes}min" for minutes in ACCESS_MINUTES], *[f"walk_same_fachgruppe_firms_{minutes}min" for minutes in (5, 10)]}
        records.append(result(year, "firm-quarter", "required schema", required.issubset(firm)))
        firm["Fachgruppe_ID"] = firm["Fachgruppe_ID"].astype(str)
        long["Fachgruppe_ID"] = long["Fachgruppe_ID"].astype(str)
        probe = firm.merge(long, left_on=["grid_id_100m","year","quarter","Fachgruppe_ID"], right_on=["grid_id","year","quarter","Fachgruppe_ID"], suffixes=("", "_grid"))
        probe = probe.merge(main[["grid_id", "year", "quarter", *[f"existing_firms_access_{minutes}min" for minutes in ACCESS_MINUTES]]], left_on=["grid_id_100m", "year", "quarter"], right_on=["grid_id", "year", "quarter"], suffixes=("", "_total_grid"))
        probe = exclude_documented_skips(probe, year)
        for minutes in ACCESS_MINUTES:
            actual = probe[f"same_fachgruppe_firms_access_{minutes}min"]
            expected = probe[f"same_fachgruppe_firms_access_{minutes}min_grid"]
            records.append(result(year, "firm-quarter", f"same-Fachgruppe {minutes}min matches cumulative long table", np.allclose(actual, expected)))
            total_actual = probe[f"existing_firms_access_{minutes}min"]
            total_expected = probe[f"existing_firms_access_{minutes}min_total_grid"]
            records.append(result(year, "firm-quarter", f"lagged-total {minutes}min matches cumulative grid table", np.allclose(total_actual, total_expected)))
        records += numeric_checks(firm, [*[f"same_fachgruppe_firms_access_{minutes}min" for minutes in ACCESS_MINUTES], *[f"walk_same_fachgruppe_firms_{minutes}min" for minutes in (5, 10)]], year, "firm-quarter")

if STATUS_PATH.exists():
    status = pd.read_csv(STATUS_PATH)
    status_years = set(pd.to_numeric(status.year, errors="coerce").dropna().astype(int)) if "year" in status else set()
    records.append(result("all", "manifest", "all years completed", status_years >= set(YEARS) and set(status.loc[status.status == "done", "year"]) >= set(YEARS)))
    records.append(result("all", "manifest", "model products use lagged firm stock", "firm_mass_source" in status and set(status.firm_mass_source.dropna()) <= {"active_firms_tminus1"}))
    records.append(result("all", "manifest", "no unresolved failed-origin manifest", not list(WORK_ROOT.glob("*/failed_origins.json"))))
    records.append(result("all", "manifest", "documented accessibility skips have year, grid, coordinates, and reason", SKIPPED_ACCESSIBILITY_PATH.exists() and {"year", "grid_id", "lat", "lon", "reason"}.issubset(skipped_accessibility) and skipped_accessibility[["year", "grid_id", "lat", "lon", "reason"]].notna().all().all()))
else:
    records.append(result("all", "manifest", "routing status exists", False))
if GRAPH_STATUS_PATH.exists():
    graphs = pd.read_csv(GRAPH_STATUS_PATH)
    graph_years = set(pd.to_numeric(graphs.year, errors="coerce").dropna().astype(int))
    manifest_rows = graphs.drop_duplicates("year", keep="last")
    records.append(result("all", "graphs", "successful graph manifests recorded for 2015-2025", graph_years >= set(YEARS) and set(manifest_rows.loc[manifest_rows.status == "done", "year"]) >= set(YEARS) and manifest_rows.manifest_path_wsl.notna().all()))
    records.append(result("all", "graphs", "pinned graph-builder image recorded", "valhalla_image_id" in graphs and set(graphs.valhalla_image_id.dropna()) == {GRAPH_VALHALLA_IMAGE_ID} and set(graphs.valhalla_image.dropna()) == {GRAPH_VALHALLA_IMAGE}))
    if "gip_snapshot_sha256" in graphs:
        expected_hashes = {year: hashlib.sha256((PBF_DIR / f"{year}.osm.pbf").read_bytes()).hexdigest() for year in YEARS if (PBF_DIR / f"{year}.osm.pbf").exists()}
        actual_hashes = graphs.drop_duplicates("year", keep="last").set_index("year")["gip_snapshot_sha256"].to_dict()
        records.append(result("all", "graphs", "graph manifests match source PBF hashes", all(actual_hashes.get(year) == digest for year, digest in expected_hashes.items()) and set(expected_hashes) >= set(YEARS)))
    else:
        records.append(result("all", "graphs", "graph manifests match source PBF hashes", False))
else:
    records.append(result("all", "graphs", "graph status exists", False))

validation = pd.DataFrame(records)
validation.to_csv(REPORT_PATH, index=False)
summary = validation.groupby(["product", "status"]).size().rename("checks").reset_index()
if (validation["status"] != "OK").any():
    failed = validation.loc[validation["status"] != "OK"].to_string(index=False)
    raise AssertionError("Routing validation failed; no analysis should start.\n" + failed)
summary